Fraud Detection Using logistic regression

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

In [13]:
df = pd.read_csv("../data/fraud_detection_paysim_dataset.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [25]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X = df[features]
y = df["isFraud"]

In [ ]:

import numpy as np

for col in ["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]:
    X[col] = np.log1p(X[col])

# check distributions
X[["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]].describe()


/var/folders/vt/534j8p4n44x12n92yxnztyy00000gn/T/ipykernel_14929/158294450.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = np.log1p(X[col])


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.084087e+01,7.414574e+00,5.366092e+00,7.722420e+00,8.330604e+00
std,1.814509e+00,5.669756e+00,6.330024e+00,6.747637e+00,6.675095e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,9.502306e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.122355e+01,9.561631e+00,0.000000e+00,1.179590e+01,1.227682e+01
75%,1.224876e+01,1.158353e+01,1.187937e+01,1.375686e+01,1.392159e+01
max,1.834213e+01,1.790292e+01,1.771920e+01,1.969049e+01,1.969094e+01


In [27]:
X = pd.get_dummies(X, columns=["type"], drop_first=True)

In [28]:
X.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9.194276,12.044359,11.984786,0.000000,0.0,False,False,True,False
1,1,7.531166,9.964112,9.872292,0.000000,0.0,False,False,True,False
2,1,5.204007,5.204007,0.000000,0.000000,0.0,False,False,False,True
3,1,5.204007,5.204007,0.000000,9.960954,0.0,True,False,False,False
4,1,9.364703,10.634773,10.305174,0.000000,0.0,False,False,True,False


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import numpy as np
print("X_train_scaled shape", X_train_scaled.shape)
print("min", np.nanmin(X_train_scaled), "max", np.nanmax(X_train_scaled))
print("mean", np.nanmean(X_train_scaled, axis=0))
print("std", np.nanstd(X_train_scaled, axis=0))
print("NaNs in scaled", np.isnan(X_train_scaled).sum())
print("Infs in scaled", np.isinf(X_train_scaled).sum())


X_train_scaled shape (5090096, 10)
min -5.973927831884512 max 12.351570248258762
mean [-1.00278164e-16 -9.02548980e-15  5.41839341e-16  8.61288584e-16
 -7.35865499e-16 -1.10736485e-15 -2.14289505e-17 -1.56602620e-17
  5.96956316e-17  3.09701450e-17]
std [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
NaNs in scaled 0
Infs in scaled 0


In [ ]:
print("y_train distribution:\n", y_train.value_counts())


y_train distribution:
 isFraud
0    5083526
1       6570
Name: count, dtype: int64


In [36]:
from sklearn.linear_model import LogisticRegression

# using solver='saga' which is more stable for large datasets and allows
# good handling of numerical issues; also make sure data is scaled/log-transformed
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1,
    solver="saga",
    penalty="l2",
    C=1.0,
    tol=1e-3
)

model.fit(X_train_scaled, y_train)

/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'saga'
,max_iter,1000
,multi_class,'deprecated'


In [37]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[[1194465   76416]
 [     42    1601]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.97      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524



Confusion matrix

[[1194465   76416]
 [     42    1601]]

 precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.97      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524


Observation: recall of 97%, 97 out of the fraud transactions marked as fraud.
Precision of 2%, only 2 out of 100 transactions marked as fraud are actually fraud. precision is very low.
Let's try higher threshold probability for marking transactions as fraud.

In [47]:
y_prob_90 = model.predict_proba(X_test_scaled)[:,1]

y_pred_90 = (y_prob_90 > 0.90).astype(int)

print(confusion_matrix(y_test, y_pred_90))
print(classification_report(y_test, y_pred_90))

[[1202485   68396]
 [     58    1585]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97   1270881
           1       0.02      0.96      0.04      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.96      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524



/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


with 90% threshold we have
confusion matrix
[[1202485   68396]
 [     58    1585]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97   1270881
           1       0.02      0.96      0.04      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.96      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524

both TP and FP have decreased and the precision is still around 2%.

In [53]:
y_prob_999 = model.predict_proba(X_test_scaled)[:,1]

y_pred_999 = (y_prob_999 > 0.999).astype(int)

print(confusion_matrix(y_test, y_pred_999))
print(classification_report(y_test, y_pred_999))
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_prob_999)

AttributeError: 'LogisticRegression' object has no attribute 'coef_'

With 99.9% probability threshold we have
confusion matrix
[[1218632   52249]
 [    114    1529]]
              precision    recall  f1-score   support

           0       1.00      0.96      0.98   1270881
           1       0.03      0.93      0.06      1643

    accuracy                           0.96   1272524
   macro avg       0.51      0.94      0.52   1272524
weighted avg       1.00      0.96      0.98   1272524

Slightly higher precision but the recall has also dropped slightly

Trying with weighted class_weight

In [54]:
from sklearn.linear_model import LogisticRegression

# using solver='saga' which is more stable for large datasets and allows
# good handling of numerical issues; also make sure data is scaled/log-transformed
modelWeighted = LogisticRegression(
    max_iter=1000,
    class_weight={0:1, 1:20},
    n_jobs=-1,
    solver="saga",
    penalty="l2",
    C=1.0,
    tol=1e-3
)

modelWeighted.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,"{0: 1, 1: 20}"
,random_state,None
,solver,'saga'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_weighted = modelWeighted.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_weighted))
print(classification_report(y_test, y_pred_weighted))


[[1269230    1651]
 [    323    1320]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.44      0.80      0.57      1643

    accuracy                           1.00   1272524
   macro avg       0.72      0.90      0.79   1272524
weighted avg       1.00      1.00      1.00   1272524



/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


With class weight, we have
confusion_matrix
[[1269230    1651]
 [    323    1320]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.44      0.80      0.57      1643

    accuracy                           1.00   1272524
   macro avg       0.72      0.90      0.79   1272524
weighted avg       1.00      1.00      1.00   1272524

Precision has improved to 44% but a significant decrease in recall now down to 80%
trying the probability threshold next

In [56]:
y_prob_90_weighted = modelWeighted.predict_proba(X_test_scaled)[:,1]

y_pred_90_weighted = (y_prob_90_weighted > 0.90).astype(int)

print(confusion_matrix(y_test, y_pred_90_weighted))
print(classification_report(y_test, y_pred_90_weighted))
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_prob_90_weighted)

/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[[1270577     304]
 [    724     919]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.75      0.56      0.64      1643

    accuracy                           1.00   1272524
   macro avg       0.88      0.78      0.82   1272524
weighted avg       1.00      1.00      1.00   1272524



0.9957490734463655

with class weight and probability threshold of 90%.
confusion matrix
[[1270577     304]
 [    724     919]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.75      0.56      0.64      1643

    accuracy                           1.00   1272524
   macro avg       0.88      0.78      0.82   1272524
weighted avg       1.00      1.00      1.00   1272524

ROC AUC score 0.9957490734463655
precision is up to 75% but recall is now at 56% 


In [57]:
y_prob_60_weighted = modelWeighted.predict_proba(X_test_scaled)[:,1]

y_pred_60_weighted = (y_prob_60_weighted > 0.60).astype(int)

print(confusion_matrix(y_test, y_pred_60_weighted))
print(classification_report(y_test, y_pred_60_weighted))
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_prob_60_weighted)

/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/himanshumehra/miniconda3/envs/fraud-detection/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[[1269852    1029]
 [    388    1255]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.55      0.76      0.64      1643

    accuracy                           1.00   1272524
   macro avg       0.77      0.88      0.82   1272524
weighted avg       1.00      1.00      1.00   1272524



0.9957490734463655